# 04 · Pianificazione e subagenti

Per compiti complessi servono due idee:
1. un **piano** esplicito che l'agente mantiene mentre lavora;
2. **subagenti**: agenti specializzati con contesto isolato, usati come tool;
3. **routing del modello**: scegliere un modello più forte solo quando serve.

## Obiettivi, prerequisiti e modalità di lettura

Renderai piano e delega osservabili, poi inserirai un router di modello. Durata: 30–40 minuti. L'euristica di routing è volutamente semplice, non una raccomandazione production.

Ogni blocco di codice è preceduto da una spiegazione e seguito da un **output
atteso**. Quando interviene un modello, l'output atteso descrive proprietà e
invarianti, non una frase letterale. Esegui le celle in ordine e non saltare i
casi negativi: mostrano il confine del meccanismo, non un incidente del corso.

## Setup (autonomo)

Ogni notebook è **indipendente**: non importa nulla dal progetto. Qui carichiamo la chiave
API dal file `.env` e creiamo un modello. Esegui le celle in ordine dall'alto verso il basso.

### Spiegazione del blocco · Setup del notebook

Configurazione e chiave vengono validate prima di creare coordinatore e subagente.

In [ ]:
# Carichiamo le variabili d'ambiente dal file `.env`.
# Lo cerchiamo nella cartella corrente e in quelle superiori, così il notebook
# funziona sia se avviato dalla radice del progetto sia dalla cartella `notebooks`.
import os
from pathlib import Path

from dotenv import load_dotenv


def trova_env() -> Path:
    for cartella in (Path.cwd(), *Path.cwd().resolve().parents):
        if (cartella / ".env").is_file():
            return cartella / ".env"
    raise FileNotFoundError("File .env non trovato: copia .env.example in .env e aggiungi la chiave.")


env_file = trova_env()
load_dotenv(env_file, override=False)          # carica le variabili senza sovrascrivere quelle già presenti
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY mancante nel file .env"
print("Ambiente caricato da:", env_file)

### Output atteso

Percorso `.env` caricato.

### Spiegazione del blocco · Modello condiviso

Il modello base alimenta coordinatore e ricercatore; il router potrà sostituirlo con un modello più forte per richieste selezionate.

In [ ]:
# `ChatOpenAI` è il wrapper LangChain attorno al modello.
# Lo creiamo una volta e lo riusiamo in tutto il notebook.
from langchain_openai import ChatOpenAI

MODELLO = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")   # modello economico, va bene per imparare
model = ChatOpenAI(
    model=MODELLO,
    use_responses_api=True,   # API "responses" di OpenAI
    store=False,              # non conservare la conversazione sui server OpenAI
)
print("Modello pronto:", MODELLO)

### Output atteso

Nome del modello base.

## 1 · Un tool per il piano

Il modo più semplice di dare "pianificazione" è un tool con cui l'agente scrive e aggiorna
una lista di passi. Il piano resta visibile e l'agente può spuntarlo.

### Spiegazione del blocco · Piano come stato esplicito

Il tool non si limita a restituire testo: aggiorna `PIANO`, rendendo i passi ispezionabili dal programma e dallo studente.

In [ ]:
from langchain_core.tools import tool

PIANO: list[str] = []


@tool
def imposta_piano(passi: list[str]) -> str:
    """Definisce l'elenco dei passi da seguire per il compito."""
    PIANO.clear()
    PIANO.extend(passi)
    return "Piano salvato:\n" + "\n".join(f"- {p}" for p in passi)

### Output atteso

Nessun output. La lista è inizialmente vuota.

## 2 · Un subagente usato come tool

Un **subagente** è un secondo agente con un compito ristretto e un proprio contesto.
Lo "impacchettiamo" dentro una funzione-tool: il main agent lo chiama come un tool qualsiasi,
ma il ragionamento del subagente resta separato (non intasa il contesto principale).

### Spiegazione del blocco · Subagente incapsulato come tool

Il ricercatore ha prompt e cronologia separati. Il coordinatore vede solo domanda e sintesi, riducendo inquinamento del contesto principale.

In [ ]:
from langchain.agents import create_agent

# Il subagente "ricercatore": un agente semplice con un compito preciso.
ricercatore = create_agent(
    model=model,
    tools=[],
    system_prompt="Sei un ricercatore. Rispondi con 3 punti concisi e concreti.",
)


@tool
def chiedi_al_ricercatore(domanda: str) -> str:
    """Delega una ricerca a un subagente specializzato e restituisce la sintesi."""
    esito = ricercatore.invoke({"messages": [{"role": "user", "content": domanda}]})
    return esito["messages"][-1].text

### Output atteso

Nessun output. `chiedi_al_ricercatore` è ora uno strumento delegabile.

## 3 · Routing del modello con un middleware

Un **middleware** si interpone attorno alla chiamata del modello. Qui scegliamo un modello
più "forte" quando la richiesta sembra complessa, altrimenti quello economico. Risparmia
soldi e latenza senza rinunciare alla qualità quando serve.

### Spiegazione del blocco · Middleware di model routing

Il middleware intercetta ogni richiesta e può sostituire il modello senza cambiare il graph. L'euristica è volutamente semplice per rendere visibile il punto di estensione.

In [ ]:
from langchain.agents.middleware import wrap_model_call

# Modello forte (facoltativo): se non configurato, riusa quello base.
model_forte = ChatOpenAI(model=os.getenv("OPENAI_STRONG_MODEL", MODELLO), use_responses_api=True, store=False)


@wrap_model_call
def instrada_modello(request, handler):
    testo = str(request.messages[-1].content).lower() if request.messages else ""
    # euristica semplice: parole "difficili" -> modello forte
    difficile = any(p in testo for p in ("architettura", "complesso", "approfondito"))
    scelto = model_forte if difficile else model
    return handler(request.override(model=scelto))

### Output atteso

Nessun output. Testi con `architettura`, `complesso` o `approfondito` useranno `model_forte`.

## 4 · Il main agent mette tutto insieme

Il coordinatore ha: il tool del piano, il subagente-come-tool e il middleware di routing.

### Spiegazione del blocco · Coordinatore

Il prompt definisce ordine operativo: pianificare, delegare quando utile, sintetizzare. Tool e middleware vengono assemblati in un unico agente.

In [ ]:
coordinatore = create_agent(
    model=model,
    tools=[imposta_piano, chiedi_al_ricercatore],
    middleware=[instrada_modello],
    system_prompt=(
        "Per compiti articolati: prima definisci un piano con imposta_piano, "
        "poi usa chiedi_al_ricercatore quando serve approfondire, infine sintetizza."
    ),
)

### Output atteso

Nessun output. `coordinatore` è pronto.

### Spiegazione del blocco · Ispezionare la traccia del run

La risposta finale nasconde i passaggi intermedi. `stampa_messaggi_run` legge
tutti i messaggi e mostra: tipo, tool chiamati (incluso il subagente), risultati
dei tool e anteprima del testo. Così vedi *come* ha lavorato il coordinatore.

### Spiegazione del blocco · Lettura della trace

L'helper rende leggibili messaggi, tool call e risultati senza nascondere deleghe. Evidenzia quando il coordinatore invoca il ricercatore e produce un riepilogo quantitativo.

In [ ]:
def stampa_messaggi_run(esito, *, max_chars: int = 240) -> None:
    """Stampa la traccia completa di un run: tipi, tool, subagenti, testo."""
    messaggi = esito["messages"] if isinstance(esito, dict) else esito
    print(f"=== Traccia run ({len(messaggi)} messaggi) ===\n")

    tools_usati: list[str] = []
    for i, m in enumerate(messaggi):
        tipo = type(m).__name__
        tool_calls = getattr(m, "tool_calls", None) or []

        if tool_calls:
            for call in tool_calls:
                nome = call["name"]
                tools_usati.append(nome)
                args = str(call.get("args", {}))
                if len(args) > max_chars:
                    args = args[:max_chars] + "…"
                sub = "  ★ SUBAGENTE" if "ricercatore" in nome else ""
                print(f"{i}. {tipo} → tool `{nome}`{sub}")
                print(f"     args: {args}")
        elif tipo == "ToolMessage":
            nome = getattr(m, "name", None) or "?"
            contenuto = str(m.content)
            if len(contenuto) > max_chars:
                contenuto = contenuto[:max_chars] + "…"
            sub = "  ★ risposta subagente" if "ricercatore" in str(nome) else ""
            print(f"{i}. {tipo} ← `{nome}`{sub}")
            print(f"     {contenuto}")
        else:
            testo = getattr(m, "text", None) or str(getattr(m, "content", ""))
            if len(testo) > max_chars:
                testo = testo[:max_chars] + "…"
            print(f"{i}. {tipo}: {testo}")

    print("\n--- Riepilogo ---")
    print("Tool invocati:", tools_usati or "(nessuno)")
    subagenti = [t for t in tools_usati if "ricercatore" in t]
    print("Subagenti scattati:", subagenti or "(nessuno)")
    print("Chiamate tool totali:", len(tools_usati))

### Output atteso

Nessun output. `stampa_messaggi_run` è pronta per mostrare trace e subagenti del run successivo.

### Output atteso

Nessun output. La funzione è pronta: la userai subito dopo l'invoke.

### Spiegazione del blocco · Esecuzione coordinata

La richiesta richiede sia piano sia delega. Alla fine si osservano risposta naturale e stato `PIANO`, due rappresentazioni complementari del lavoro.

In [ ]:
esito = coordinatore.invoke({"messages": [{
    "role": "user",
    "content": "Prepara un mini piano per valutare librerie Python e delega la ricerca. le librerie sono sklearn e pytorch. Restituiscimi un'analisi approfondita delle due librerie",
}]})

stampa_messaggi_run(esito)
print("\n=== Risposta finale ===")
print(esito["messages"][-1].text)
print("\nPiano registrato:", PIANO)

### Output atteso

Una sintesi sulle due librerie e una lista di passi registrata. Dettagli dipendono dal modello.

## Prova tu

- Aggiungi un subagente "revisore" che critica il risultato prima della conclusione.
- Estendi `stampa_messaggi_run` così marca anche il revisore come ★ SUBAGENTE.

**Idea chiave**: i subagenti isolano il contesto (meno rumore) e il routing usa il modello
giusto al momento giusto. Sono i mattoni per scalare la complessità.

## Laboratorio aggiuntivo

Gli esempi seguenti riusano quanto costruito sopra. Il primo amplia il caso normale; il
secondo esercita un confine, un errore o una proprietà che spesso causa bug reali.

## Esempio aggiuntivo: piano verificabile senza agente

### Spiegazione del blocco

Testare il tool direttamente mostra che il piano è vero stato applicativo, non solo testo generato.

In [ ]:
print(imposta_piano.invoke({"passi": ["raccogli criteri", "confronta opzioni", "verifica conclusione"]}))
assert PIANO[-1] == "verifica conclusione"
print("Passi registrati:", len(PIANO))

### Output atteso

Tre righe del piano e `Passi registrati: 3`; l'assert deve passare.

## Esempio aggiuntivo: delega diretta e contesto isolato

### Spiegazione del blocco

Invocare il tool-subagente separatamente rende visibile il suo contratto: una domanda entra, una sintesi ristretta esce.

In [ ]:
sintesi = chiedi_al_ricercatore.invoke({
    "domanda": "Indica tre criteri per confrontare librerie Python, senza scegliere una libreria."
})
print(sintesi)

### Output atteso

Tre punti concisi, per esempio manutenzione, API e prestazioni. Il testo varia e comporta una chiamata modello.

## Riepilogo e troubleshooting

Prima di proseguire, prova a spiegare con parole tue: quale stato è cambiato, quale
componente ha preso la decisione e quale prova rende osservabile l'esito.

Se una cella fallisce:

1. rileggi l'output atteso e individua la prima invariante non rispettata;
2. verifica di aver eseguito tutte le celle precedenti nello stesso kernel;
3. per i notebook live, controlla `.env`, modello disponibile e quota API;
4. riavvia il kernel solo dopo aver conservato eventuali file che vuoi ispezionare;
5. non correggere un caso negativo: l'errore previsto è parte dell'esempio.